In [6]:
import re
import math
from collections import Counter
from sklearn.model_selection import train_test_split

# 1. Corpus
corpus = [
    "natural language processing is interesting",
    "language models predict the next word",
    "natural language models learn from text",
    "machine learning improves language processing",
    "the model predicts words using context",
    "language processing uses statistical models",
    "machine learning learns from data",
    "the language model predicts the next word",
    "natural language processing uses machine learning",
    "statistical language models predict text"
]

# 2. Preprocessing
def preprocess(text):
    return re.findall(r"\b\w+\b", text.lower())

sentences = [preprocess(s) for s in corpus]

# Split data
train, test = train_test_split(
    sentences, test_size=0.2, random_state=42
)

# Display dataset information
print("Total sentences:", len(sentences))
print("Training sentences:", len(train))
print("Testing sentences:", len(test))
print("Sample:", sentences[0])


# 3. N-gram Model
class NGramModel:

    def __init__(self, n, data):
        self.n = n
        self.ngrams = Counter()
        self.contexts = Counter()
        self.vocab = {"<UNK>", "<s>", "</s>"}

        for s in data:
            self.vocab.update(s)

        self.V = len(self.vocab)

        for s in data:
            tokens = ["<s>"] * (n-1) + s + ["</s>"]

            for i in range(n-1, len(tokens)):
                gram = tuple(tokens[i-n+1:i+1])
                self.ngrams[gram] += 1
                self.contexts[gram[:-1]] += 1

    # 4. Probability
    def probability(self, gram):
        gram = tuple(gram)
        return (self.ngrams[gram] + 1) / (
            self.contexts[gram[:-1]] + self.V
        )

    # 5. Sentence Probability
    def sentence_probability(self, sentence):
        tokens = ["<s>"] * (self.n-1) + sentence + ["</s>"]
        log_p = 0
        count = 0

        for i in range(self.n-1, len(tokens)):
            gram = tuple(tokens[i-self.n+1:i+1])
            log_p += math.log(self.probability(gram))
            count += 1

        return log_p, count


# 6. Train Models
models = {n: NGramModel(n, train) for n in [1, 2, 3]}

print("\nUnigram, Bigram and Trigram models trained!")


# 7. Perplexity Evaluation
for n, model in models.items():
    total_log = 0
    total_words = 0

    for s in test:
        log_p, count = model.sentence_probability(s)
        total_log += log_p
        total_words += count

    ppl = math.exp(-total_log / total_words)
    print(n, "-gram Perplexity:", round(ppl, 4))


# 8. Domain Adaptation
custom_corpus = [
    "language models predict words based on context",
    "machine learning improves text prediction",
    "statistical models learn language patterns"
]

custom_data = [preprocess(s) for s in custom_corpus]

adapted_models = {
    n: NGramModel(n, custom_data) for n in [1, 2, 3]
}

print("\nDomain adaptation completed!")

Total sentences: 10
Training sentences: 8
Testing sentences: 2
Sample: ['natural', 'language', 'processing', 'is', 'interesting']

Unigram, Bigram and Trigram models trained!
1 -gram Perplexity: 21.3196
2 -gram Perplexity: 15.3711
3 -gram Perplexity: 19.1751

Domain adaptation completed!
